In [39]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from gensbi_test.functions import power_law_spectrum
from gensbi_test.functions import generate_dummy_maps

In [40]:
# Introducing tokens: a 16x16=256 pixel map, in a sequence of 8 energy bins. 
# I use 256 tokens, one per pixel. Each token contains 8 photon counts, one per energy bin (dim 8). It also contains information
#  about pixel's location in 2D (dim 2), exposure (dim 8), galactic foreground information (dim 8). Total dimension of each token
#  is 8+2+8+8=26. The sequence length is 256, the number of tokens.
# !! could be a good idea to add a token for the global information, like the total exposure, the mean energy per bin, etc. !!

# data structure
n_healPIX = 1
n_side = 16
n_tokens = n_side * n_side * n_healPIX
n_energy = 8 # number of energy bins
n_energy_features = 3  # photon counts, exposure, galactic foreground 
pixel_coordinates = 2
d_model = n_energy * n_energy_features + pixel_coordinates 

# energy bins
Emin = 1.0  # GeV
Emax = 10.0 # GeV
theta = torch.tensor([1.2, 0.8, 1.5])

# Neural network hyperparameters
batch_size = 4
n_heads = 1 




In [59]:
# Generate first training batch
maps = [generate_dummy_maps(n_side, Emin, Emax, n_energy, theta) for b in range(batch_size)]

# Proceed with tokenization: I will get a full_batch tensor with dimension 4 , 256 , 26
# each pixel (token of choice) has 26 features: 2 coordinates, 8 energy bin photon counts values, 8 galactic foreground values, 8 exposure values
grid_batch = torch.stack([m[0] for m in maps], dim=0)
exposure_batch = torch.stack([m[1] for m in maps], dim=0) 
foreground_batch = torch.stack([m[2] for m in maps], dim=0) 
photon_counts_batch = torch.stack([m[3] for m in maps], dim=0) 
full_batch = torch.cat( (grid_batch,exposure_batch,foreground_batch,photon_counts_batch), dim =-1 ).flatten(1,2)

exposure_batch.size()

torch.Size([4, 16, 16, 8])

4